#### MODEL BASE CONTENT FILTERING

#### UCITALI DS

In [94]:

import torch
torch.cuda.set_per_process_memory_fraction(0.18)



In [122]:
import pandas as pd
df_train = pd.read_csv("df_trainveci2.csv")
df_train = df_train.sort_values(by=['user_id', 'play_count'], ascending=[True, False])
df_train = df_train.reset_index(drop=True)


df_test = pd.read_csv("df_test2.csv")
df_test = df_test.sort_values(by=['user_id', 'play_count'], ascending=[True, False])
df_test = df_test.reset_index(drop=True)

df_val = pd.read_csv("df_val2.csv")
df_val = df_val.sort_values(by=['user_id', 'play_count'], ascending=[True, False])
df_val = df_val.reset_index(drop=True)

#### dodaje se mode posto nema u ds

In [123]:
# import pandas as pd

# KEYS = ["artist_id", "song_id", "user_id"]

# df = pd.read_csv('mergeovan.csv')[KEYS + ["mode"]]

# df_train = df_train.merge(df, on=KEYS, how="left")
# df_test  = pd.read_csv('df_test.csv').merge(df, on=KEYS, how="left")
# df_val   = pd.read_csv('df_val.csv').merge(df, on=KEYS, how="left")

# df_train.head()


In [124]:
import pandas as pd
import os
import h5py

df_ceo = pd.read_csv('mergeovan.csv')
df= df_ceo[['user_id', 'song_id', 'play_count', 'artist_id', 'artist_terms', 'loudness', 'tempo', 'time_signature','mode', 'similar_artists']]

df.sample()

,user_id,song_id,play_count,artist_id,artist_terms,loudness,tempo,time_signature,mode,similar_artists
15510,0028292aa536122c1f86fd48a39bd83fe582d27f,SOLLOTO12AB01804C6,10,ARRDRI71187FB46EAA,"['folk-pop', 'folk', 'rock', 'pop', 'indie']",0.713881,0.323417,0.571429,1.0,"['ARLNQBR1187B9B9C4D', 'ARU3BQ91187FB5A3D6', '..."


In [125]:
#prazna polja

numericke_kolone = ['loudness', 'tempo', 'time_signature','play_count', 'mode']
print(df[numericke_kolone].isna().sum())
print("-" * 40)

for kolona in numericke_kolone:
    prazni_redovi = df[df[kolona].isna()].index.tolist()
    
    if len(prazni_redovi) > 0:
        print(f"Kolona '{kolona}' ima prazna polja u ")
        if len(prazni_redovi) > 20:
            print(f"  {prazni_redovi[:20]} {len(prazni_redovi) - 20} redova)")
        else:
            print(f"  {prazni_redovi}")
    else:
        print(f"Kolona '{kolona}' nema")
print(len(df))

loudness          0
tempo             0
time_signature    0
play_count        0
mode              0
dtype: int64
----------------------------------------
Kolona 'loudness' nema
Kolona 'tempo' nema
Kolona 'time_signature' nema
Kolona 'play_count' nema
Kolona 'mode' nema
15695


In [126]:
# df_train = pd.read_csv('df_trainveci2.csv')
# df_test = pd.read_csv('df_test2.csv')
# df_val = pd. read_csv('df_val2.csv')

In [127]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

tfidf = TfidfVectorizer(max_features=5)



df['artist_terms'] = df['artist_terms'].fillna('unknown')
df['terms_clean_text'] = df['artist_terms'].astype(str).str.lower()
df['terms_clean_text'] = df['terms_clean_text'].str.replace(r"[\[\]'\"]", "", regex=True)
df['terms_clean_text'] = df['terms_clean_text'].apply(lambda x: 'unknown' if x.strip() == '' else x)

tfidf = TfidfVectorizer(max_features=1000)
tfidf_matrica = tfidf.fit_transform(df['terms_clean_text'])




df_test['artist_terms'] = df_test['artist_terms'].fillna('unknown')
df_test['terms_clean_text'] = df_test['artist_terms'].astype(str).str.lower()
df_test['terms_clean_text'] = df_test['terms_clean_text'].str.replace(r"[\[\]'\"]", "", regex=True)
df_test['terms_clean_text'] = df_test['terms_clean_text'].apply(lambda x: 'unknown' if x.strip() == '' else x)

tfidf = TfidfVectorizer(max_features=1000)
tfidf_matrica_test = tfidf.fit_transform(df_test['terms_clean_text'])

print(f"pesme x karakteristike {tfidf_matrica_test.shape}") 

df_train['artist_terms'] = df_train['artist_terms'].fillna('unknown')
df_train['terms_clean_text'] = df_train['artist_terms'].astype(str).str.lower()
df_train['terms_clean_text'] = df_train['terms_clean_text'].str.replace(r"[\[\]'\"]", "", regex=True)
df_train['terms_clean_text'] = df_train['terms_clean_text'].apply(lambda x: 'unknown' if x.strip() == '' else x)

tfidf = TfidfVectorizer(max_features=1000)
tfidf_matrica_train = tfidf.fit_transform(df_train['terms_clean_text'])

print(f"pesme x karakteristike {tfidf_matrica_train.shape}") 

df_val['artist_terms'] = df_val['artist_terms'].fillna('unknown')
df_val['terms_clean_text'] = df_val['artist_terms'].astype(str).str.lower()
df_val['terms_clean_text'] = df_val['terms_clean_text'].str.replace(r"[\[\]'\"]", "", regex=True)
df_val['terms_clean_text'] = df_val['terms_clean_text'].apply(lambda x: 'unknown' if x.strip() == '' else x)

tfidf = TfidfVectorizer(max_features=1000)
tfidf_matrica_val = tfidf.fit_transform(df_val['terms_clean_text'])

print(f"pesme x karakteristike {tfidf_matrica_val.shape}") 

pesme x karakteristike (54319, 428)
pesme x karakteristike (200339, 455)
pesme x karakteristike (40419, 382)


In [128]:
df['similar_artists'] = df['similar_artists'].fillna('unknown')

df['artists_clean_text'] = df['similar_artists'].astype(str).str.lower()
df['artists_clean_text'] = df['artists_clean_text'].str.replace(r"[\[\]'\"]", "", regex=True)

df['artists_clean_text'] = df['artists_clean_text'].apply(lambda x: 'unknown' if x.strip() == '' else x)

tfidf = TfidfVectorizer(max_features=1000)
tfidf_matrica = tfidf.fit_transform(df['artists_clean_text'])

print(f"pesme x karakteristike {tfidf_matrica.shape}") 



df_test['similar_artists'] = df_test['similar_artists'].fillna('unknown')

df_test['artists_clean_text'] = df_test['similar_artists'].astype(str).str.lower()
df_test['artists_clean_text'] = df_test['artists_clean_text'].str.replace(r"[\[\]'\"]", "", regex=True)

df_test['artists_clean_text'] = df_test['artists_clean_text'].apply(lambda x: 'unknown' if x.strip() == '' else x)

tfidf_matrica_test = tfidf.fit_transform(df_test['artists_clean_text'])

print(f"pesme x karakteristike {tfidf_matrica_test.shape}") 



df_train['similar_artists'] = df_train['similar_artists'].fillna('unknown')

df_train['artists_clean_text'] = df_train['similar_artists'].astype(str).str.lower()
df_train['artists_clean_text'] = df_train['artists_clean_text'].str.replace(r"[\[\]'\"]", "", regex=True)
df_train['artists_clean_text'] = df_train['artists_clean_text'].apply(lambda x: 'unknown' if x.strip() == '' else x)

tfidf_matrica_train = tfidf.fit_transform(df_train['artists_clean_text'])

print(f"pesme x karakteristike {tfidf_matrica_train.shape}") 

df_val['similar_artists'] = df_val['similar_artists'].fillna('unknown')
df_val['artists_clean_text'] = df_val['similar_artists'].astype(str).str.lower()
df_val['artists_clean_text'] = df_val['artists_clean_text'].str.replace(r"[\[\]'\"]", "", regex=True)
df_val['artists_clean_text'] = df_val['artists_clean_text'].apply(lambda x: 'unknown' if x.strip() == '' else x)

tfidf_matrica_val = tfidf.fit_transform(df_val['terms_clean_text'])

print(f"pesme x karakteristike {tfidf_matrica_val.shape}") 

pesme x karakteristike (15695, 1000)
pesme x karakteristike (54319, 1000)
pesme x karakteristike (200339, 1000)
pesme x karakteristike (40419, 382)


In [129]:
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

def ocisti_tekst(dataframe, izvorna_kolona, nova_kolona):
    # Popunjavanje praznih vrednosti sa 'unknown'
    dataframe[izvorna_kolona] = dataframe[izvorna_kolona].fillna('unknown')
    # Mala slova i pretvaranje u string
    dataframe[nova_kolona] = dataframe[izvorna_kolona].astype(str).str.lower()
    # Uklanjanje zagrada i navodnika
    dataframe[nova_kolona] = dataframe[nova_kolona].str.replace(r"[\[\]'\"]", "", regex=True)
    # Provera da li je ostao prazan string nakon čišćenja
    dataframe[nova_kolona] = dataframe[nova_kolona].apply(lambda x: 'unknown' if x.strip() == '' else x)
    return dataframe

# --- KORAK 2: Primenjujemo čišćenje na sve skupove ---
# Čistimo 'artist_terms' -> 'terms_clean_text'
df_train = ocisti_tekst(df_train, 'artist_terms', 'terms_clean_text')
df_val = ocisti_tekst(df_val, 'artist_terms', 'terms_clean_text')
df_test = ocisti_tekst(df_test, 'artist_terms', 'terms_clean_text')

# --- KORAK 3: Ispravan TF-IDF (Sprečavanje Data Leakage-a) ---
# Kreiramo JEDAN vectorizer sa maksimalno 1000 reči
tfidf = TfidfVectorizer(max_features=5)

# Povezujemo rečnik (FIT) i transformišemo SAMO trening skup
tfidf_matrica_train = tfidf.fit_transform(df_train['terms_clean_text'])

# Test i Validation SAMO transformišemo (koriste rečnik iz treninga!)
tfidf_matrica_val = tfidf.transform(df_val['terms_clean_text'])
tfidf_matrica_test = tfidf.transform(df_test['terms_clean_text'])

# Provera dimenzija (broj kolona mora biti tačno 1000 za sve!)
print(f"Train matrica: {tfidf_matrica_train.shape}") 
print(f"Validation matrica: {tfidf_matrica_val.shape}") 
print(f"Test matrica: {tfidf_matrica_test.shape}")

Train matrica: (200339, 5)
Validation matrica: (40419, 5)
Test matrica: (54319, 5)


#### delimo na test i train set

In [130]:
numericke_kolone = ['loudness', 'tempo', 'time_signature', 'play_count','mode', 'terms_clean_text', 'artists_clean_text']
print(df[numericke_kolone].isna().sum())
print("-" * 40)

for kolona in numericke_kolone:
    prazni_redovi = df[df[kolona].isna()].index.tolist()
    
    if len(prazni_redovi) > 0:
        print(f"kolons '{kolona}' ima prazno ")
        if len(prazni_redovi) > 20:
            print(f"  {prazni_redovi[:20]} {len(prazni_redovi) - 20} redova)")
        else:
            print(f"  {prazni_redovi}")
    else:
        print(f"kolomns '{kolona}' nema")

loudness              0
tempo                 0
time_signature        0
play_count            0
mode                  0
terms_clean_text      0
artists_clean_text    0
dtype: int64
----------------------------------------
kolomns 'loudness' nema
kolomns 'tempo' nema
kolomns 'time_signature' nema
kolomns 'play_count' nema
kolomns 'mode' nema
kolomns 'terms_clean_text' nema
kolomns 'artists_clean_text' nema


In [89]:
print(df.columns.tolist())

['user_id', 'song_id', 'play_count', 'artist_id', 'artist_terms', 'loudness', 'tempo', 'time_signature', 'mode', 'similar_artists', 'terms_clean_text', 'artists_clean_text']


## DECISION TREE

In [80]:
from sklearn import tree
from sklearn.preprocessing import LabelEncoder

# 1. Kreiramo Classifier (klasifikator) umesto Regressor-a
clf = tree.DecisionTreeClassifier(random_state=42)

# 2. Inicijalizujemo LabelEncoder
le = LabelEncoder()

# 3. Fitujemo encoder na svim ID-jevima pesama da nauči rečnik pesama
sve_pesme = pd.concat([df_test['song_id'], df_train['song_id'], df_val['song_id']])
le.fit(sve_pesme)

# 4. Pretvaramo tekstualne oznake u brojeve (0, 1, 2...) za y_train, y_test, y_val
y_train = le.transform(df_train['song_id'])
y_test = le.transform(df_test['song_id'])
y_val = le.transform(df_val['song_id'])

# 5. Izdvajamo numeričke karakteristike za X
x_train = df_train[['loudness', 'tempo', 'time_signature', 'play_count', 'mode']]
x_test = df_test[['loudness', 'tempo', 'time_signature', 'play_count', 'mode']]
x_val = df_val[['loudness', 'tempo', 'time_signature', 'play_count', 'mode']]

# 6. Pokrećemo fitovanje - sada više neće biti greške!
clf = clf.fit(x_train, y_train)
print("Model je uspešno istreniran!")

Model je uspešno istreniran!


In [29]:
# le = LabelEncoder()
#k = le.fit_transform(df['song_id'])

# clf.predict()

# tree.plot_tree(clf)

In [54]:
print(df_train.columns.tolist())

['user_id', 'song_id', 'play_count', 'artist_id', 'artist_terms', 'loudness', 'mode', 'tempo', 'time_signature', 'similar_artists', 'terms_clean_text', 'artists_clean_text']


In [55]:
print("Redova u tfidf_matrica:", tfidf_matrica.shape[0])
print("Jedinstvenih song_id u train:", df_train['song_id'].nunique())
print("Duzina song_id_u_indeks:", len(song_id_u_indeks))

Redova u tfidf_matrica: 15695
Jedinstvenih song_id u train: 2262
Duzina song_id_u_indeks: 1580


In [30]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import LabelEncoder
# from scipy.sparse import hstack, csr_matrix  
# import numpy as np

# tfidf = TfidfVectorizer(max_features=1000, stop_words='english') 
# tekst_za_tfidf = df['artists_clean_text'].astype(str) + " " + df['terms_clean_text'].astype(str)
# tfidf_matrica = tfidf.fit_transform(tekst_za_tfidf)

# le = LabelEncoder()
# y = le.fit_transform(df['song_id'])

# x_numericki = df[['loudness', 'tempo', 'time_signature', 'mode']].values

# x_numericki_sparse = csr_matrix(x_numericki) 


# x_train=df_train[['loudness', 'tempo', 'time_signature', 'play_count', 'mode']]
# y_train = df_train['song_id']

# x_test=df_test[['loudness', 'tempo', 'time_signature', 'play_count', 'mode']]
# y_test = df_test['song_id']

# x_val=df_val[['loudness', 'tempo', 'time_signature', 'play_count', 'mode']]
# y_val = df_val['song_id']





# # x = hstack([x_numericki_sparse, tfidf_matrica], format='csr')

# # x, x_test, y, y_test = train_test_split(x, y, test_size=0.15, random_state=42)
# # x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.1765, random_state=42)

# clf = tree.DecisionTreeClassifier(max_depth=20)
# clf = clf.fit(x_train, y_train)
# y_pred = clf.predict(x_val)
# hitovi = (y_pred == y_val).sum()
# print(f" {(hitovi / len(y_val)) * 100:.2f}%")

 95.90%


In [39]:
df.head()
jedinstvene_pesme = df[['song_id', 'artist_id']].drop_duplicates().reset_index(drop=True)

song_id_u_indeks = {}
for indeks, red in jedinstvene_pesme.iterrows():
    song_id_u_indeks[red['song_id']] = indeks

print(len(song_id_u_indeks))


1580


In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer

# --- 1. Jedinstvene pesme (jedan red po song_id) ---
jedinstvene_pesme = df[['song_id', 'artist_id', 'terms_clean_text']].drop_duplicates(
    subset='song_id'
).reset_index(drop=True)

print("Broj jedinstvenih pesama:", len(jedinstvene_pesme))

# --- 2. song_id_u_indeks PRAVI SE IZ ISTOG dataframe-a, istim redosledom ---
song_id_u_indeks = {
    s_id: idx for idx, s_id in enumerate(jedinstvene_pesme['song_id'])
}

# --- 3. tfidf_matrica se pravi iz ISTOG jedinstvene_pesme dataframe-a, istim redosledom ---
# (NE iz df['terms_clean_text'] direktno - to ima duplikate i drugaciji redosled!)
tfidf = TfidfVectorizer(max_features=1000)
tfidf_matrica = tfidf.fit_transform(jedinstvene_pesme['terms_clean_text'])

print("Dimenzije tfidf_matrica:", tfidf_matrica.shape)
print("Duzina song_id_u_indeks:", len(song_id_u_indeks))
# Ova dva broja MORAJU biti ista (broj redova u tfidf_matrica == duzina song_id_u_indeks)
assert tfidf_matrica.shape[0] == len(song_id_u_indeks), "Indeksi se ne poklapaju!"
print("OK - indeksi se poklapaju.")

Broj jedinstvenih pesama: 1580
Dimenzije tfidf_matrica: (1580, 431)
Duzina song_id_u_indeks: 1580
OK - indeksi se poklapaju.


In [32]:
print(df_train.shape[0])

200339


### ZA MASINU

In [57]:
import time
import numpy as np

def evaluiraj_maksimalni_dt_DEBUG(
    df_train,
    df_test,
    tfidf_matrica,
    song_id_u_indeks,
    df_pesme,
    top_n=15,
    max_korisnika=5,   # samo prvih N korisnika da vidimo brzinu
):
    test_users = df_test["user_id"].unique()
    print("ukupno test_users:", len(test_users))
    print("dimenzije tfidf_matrica:", tfidf_matrica.shape)

    tfidf_matrica = tfidf_matrica.tocsr()
    broj_pesama = tfidf_matrica.shape[0]

    indeks_u_song_id = np.empty(broj_pesama, dtype=object)
    for s_id, idx in song_id_u_indeks.items():
        indeks_u_song_id[idx] = s_id

    svi_indeksi_pesama = np.arange(broj_pesama)

    from sklearn import tree

    for i, target_user_id in enumerate(test_users[:max_korisnika]):
        t_start = time.time()

        user_test_redovi = df_test[df_test["user_id"] == target_user_id]
        user_train = df_train[df_train["user_id"] == target_user_id]
        if user_test_redovi.empty or user_train.empty:
            continue

        skup_slusanih_id = set(user_train["song_id"])
        pozitivni_indeksi = [
            song_id_u_indeks[s_id] for s_id in user_train["song_id"]
            if s_id in song_id_u_indeks
        ]
        if not pozitivni_indeksi:
            continue

        t_priprema = time.time()

        pozitivni_set = set(pozitivni_indeksi)
        broj_negativnih = len(pozitivni_indeksi) * 2
        maska_dozvoljenih = np.ones(broj_pesama, dtype=bool)
        maska_dozvoljenih[list(pozitivni_set)] = False
        kandidati = svi_indeksi_pesama[maska_dozvoljenih]
        broj_negativnih = min(broj_negativnih, len(kandidati))
        negativni_indeksi = np.random.choice(kandidati, size=broj_negativnih, replace=False).tolist()

        t_sampling = time.time()

        idx_trening = pozitivni_indeksi + negativni_indeksi
        X = tfidf_matrica[idx_trening]
        y = np.array([1]*len(pozitivni_indeksi) + [0]*len(negativni_indeksi))

        clf = tree.DecisionTreeClassifier(max_depth=1000, random_state=42)
        clf.fit(X, y)

        t_fit = time.time()

        skorovi = clf.predict_proba(tfidf_matrica)[:, 1]

        t_predict = time.time()

        top_idx = np.argsort(skorovi)[::-1]

        t_sort = time.time()

        print(f"--- korisnik {i} ({target_user_id}) ---")
        print(f"  priprema (filter df):  {t_priprema - t_start:.4f}s")
        print(f"  sampling negativnih:   {t_sampling - t_priprema:.4f}s")
        print(f"  fit stabla:            {t_fit - t_sampling:.4f}s   (broj_pozitivnih={len(pozitivni_indeksi)})")
        print(f"  predict_proba (sve):   {t_predict - t_fit:.4f}s")
        print(f"  argsort:               {t_sort - t_predict:.4f}s")
        print(f"  UKUPNO:                {t_sort - t_start:.4f}s")



In [81]:
import numpy as np
import pandas as pd
from sklearn import tree
import scipy.sparse as sp


def evaluiraj_maksimalni_dt(
    df_train,
    df_test,
    tfidf_matrica,
    song_id_u_indeks,
    df_pesme,
    top_n=15,
):
    ukupno_hitova = 0
    ukupno_preciznost = 0.0
    procenjeni_korisnici = 0

    test_users = df_test["user_id"].unique()
    tfidf_matrica = tfidf_matrica.tocsr()
    broj_pesama = tfidf_matrica.shape[0]

    indeks_u_song_id = np.empty(broj_pesama, dtype=object)
    for s_id, idx in song_id_u_indeks.items():
        indeks_u_song_id[idx] = s_id

    svi_indeksi_pesama = np.arange(broj_pesama)

    for target_user_id in test_users:
        user_test_redovi = df_test[df_test["user_id"] == target_user_id]
        if user_test_redovi.empty:
            continue

        user_train = df_train[df_train["user_id"] == target_user_id]
        if user_train.empty:
            continue

        user_test_pesme = set(user_test_redovi["song_id"])
        skup_slusanih_id = set(user_train["song_id"])

        pozitivni_indeksi = [
            song_id_u_indeks[s_id]
            for s_id in user_train["song_id"]
            if s_id in song_id_u_indeks
        ]
        if not pozitivni_indeksi:
            continue

        pozitivni_set = set(pozitivni_indeksi)

        # --- VEKTORIZOVANO negativno sampling-ovanje, bez while petlje ---
        broj_negativnih = len(pozitivni_indeksi) * 2

        # kandidati = svi indeksi OSIM onih koje je korisnik vec slusao
        # (radi se preko mask-a, brzo i bez rizika od infinite loop-a)
        maska_dozvoljenih = np.ones(broj_pesama, dtype=bool)
        maska_dozvoljenih[list(pozitivni_set)] = False

        # ako slusane pesme imaju indekse i van pozitivni_set (npr. dupli song_id),
        # ovo ih ne hvata, ali pozitivni_set vec pokriva sve s_id iz song_id_u_indeks
        kandidati = svi_indeksi_pesama[maska_dozvoljenih]

        if len(kandidati) == 0:
            continue

        broj_negativnih = min(broj_negativnih, len(kandidati))
        negativni_indeksi = np.random.choice(
            kandidati, size=broj_negativnih, replace=False
        ).tolist()

        trenutni_indeksi_trening = pozitivni_indeksi + negativni_indeksi
        X_user_train = tfidf_matrica[trenutni_indeksi_trening]
        y_user_train = np.array(
            [1] * len(pozitivni_indeksi) + [0] * len(negativni_indeksi)
        )

        clf = tree.DecisionTreeClassifier(max_depth=10, random_state=42)
        clf.fit(X_user_train, y_user_train)

        # predikcija za sve pesme odjednom (predict_proba radi i nad sparse celom matricom,
        # batch-ovanje nije neophodno za DecisionTree, ali ostavljeno za velike kataloge)
        skorovi = clf.predict_proba(tfidf_matrica)[:, 1]

        top_indeksi = np.argsort(skorovi)[::-1]

        preporuke_id = []
        vec_dodato = set()

        for idx in top_indeksi:
            s_id = indeks_u_song_id[idx]

            if s_id not in skup_slusanih_id and s_id not in vec_dodato:
                preporuke_id.append(s_id)
                vec_dodato.add(s_id)

            if len(preporuke_id) == top_n:
                break

        pogoci_set = user_test_pesme.intersection(preporuke_id)
        broj_pogodaka = len(pogoci_set)

        if broj_pogodaka > 0:
            ukupno_hitova += 1
        ukupno_preciznost += broj_pogodaka / top_n
        procenjeni_korisnici += 1

    prosecan_hr = (
        ukupno_hitova / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    )
    prosecna_preciznost = (
        ukupno_preciznost / procenjeni_korisnici
        if procenjeni_korisnici > 0
        else 0
    )

    print(f"evaluirano korisnika drvce {procenjeni_korisnici}")
    print(f"Hit Rate@{top_n}:     {prosecan_hr:.4f}")
    print(f"Precision@{top_n}:    {prosecna_preciznost:.4f}")

    return prosecan_hr, prosecna_preciznost

hr_full, prec_full = evaluiraj_maksimalni_dt(
    df_train=df_train,
    df_test=df_test,  
    tfidf_matrica=tfidf_matrica, 
    song_id_u_indeks=song_id_u_indeks,
    df_pesme=df,
    top_n=5,
)

evaluirano korisnika drvce 14127
Hit Rate@5:     0.0028
Precision@5:    0.0006


## JACCARD SIM

In [131]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

izvor_interakcija = df  

# train_list = []
# test_list = []

# for user_id, group in izvor_interakcija.groupby("user_id"):
#     n_samples = len(group)

#     if n_samples >= 6:
#         u_train, u_test = train_test_split(
#            group, test_size=0.30, shuffle=True, random_state=42
#         )
#         train_list.append(u_train)
#         test_list.append(u_test)

#     elif 3 <= n_samples <= 5:
#         group_shuffled = group.sample(frac=1, random_state=42)
#         u_test = group_shuffled.iloc[[1]]
#         u_train = group_shuffled.iloc[2:]
#         train_list.append(u_train)
#         test_list.append(u_test)

#     elif n_samples == 2:
#         u_train, u_test = train_test_split(
#             group, test_size=0.50, shuffle=True, random_state=42
#         )
#         train_list.append(u_train)
#         test_list.append(u_test)
    
#     else:
#         train_list.append(group)

# df_train = pd.concat(train_list, ignore_index=True)
# df_test = pd.concat(test_list, ignore_index=True)

# print(f"Veličina train seta:       {df_train.shape[0]} interakcija")
# print(f"Veličina test seta:        {df_test.shape[0]} interakcija")

# df_jedinstvene_pesme = df[['song_id', 'artists_clean_text', 'terms_clean_text']].drop_duplicates(subset=['song_id']).reset_index(drop=True)
# print(f"Ukupno jedinstvenih pesama za preporuku: {df_jedinstvene_pesme.shape[0]}\n")


# df_train = pd.read_csv(r'C:\Users\Lena\Downloads\millionsongsubset\df_train.csv')
# df_test = pd.read_csv(r'C:\Users\Lena\Downloads\millionsongsubset\df_test.csv')
# df_val = pd. read_csv(r'C:\Users\Lena\Downloads\millionsongsubset\df_val.csv')

# x_train=df_train[['loudness', 'tempo', 'time_signature', 'play_count']]
# y_train = df_train['song_id']

# x_test=df_test[['loudness', 'tempo', 'time_signature', 'play_count']]
# y_test = df_test['song_id']

# x_val=df_val[['loudness', 'tempo', 'time_signature', 'play_count']]
# y_val = df_val['song_id']

def evaluiraj_maksimalni_jaccard(
    df_train,
    df_test,
    tfidf_matrica,      
    song_id_u_indeks,    
    df_pesme,
    top_n=15,
):
    # 1. INICIJALIZACIJA (Dodate MAP i NDCG promenljive ovde)
    ukupno_hitova = 0
    ukupno_preciznost = 0.0
    ukupno_ap = 0.0          
    ukupno_ndcg = 0.0        
    procenjeni_korisnici = 0

    test_users = df_test["user_id"].unique()

    print("Pripremam skupove reči za sve pesme...")
    pesme_skupovi_reci = []
    for _, red in df_pesme.iterrows():
        tekst = str(red['artists_clean_text']) + " " + str(red['terms_clean_text'])
        pesme_skupovi_reci.append(set(tekst.split()))
    
    broj_pesama = len(pesme_skupovi_reci)
    brzi_maping = {red['song_id']: i for i, red in df_pesme.iterrows()}

    print("Započinjem evaluaciju korisnika...")
    for target_user_id in test_users:
        user_test_redovi = df_test[df_test["user_id"] == target_user_id]
        if user_test_redovi.empty:
            continue
            
        user_train = df_train[df_train["user_id"] == target_user_id]
        if user_train.empty:
            continue

        user_test_pesme = set(user_test_redovi["song_id"])
        skup_slusanih_id = set(user_train["song_id"])

        # Profil korisnika
        profil_korisnika_skup = set()
        for s_id in user_train["song_id"]:
            if s_id in brzi_maping:
                idx_p = brzi_maping[s_id]
                profil_korisnika_skup.update(pesme_skupovi_reci[idx_p])

        if not profil_korisnika_skup:
            continue

        # Jaccard
        skorovi = np.zeros(broj_pesama)
        for idx_p in range(broj_pesama):
            skup_pesme = pesme_skupovi_reci[idx_p]
            presek = len(profil_korisnika_skup.intersection(skup_pesme))
            unija = len(profil_korisnika_skup)+len(skup_pesme)-presek
            skorovi[idx_p] = presek / unija if unija > 0 else 0

        top_indeksi = np.argsort(skorovi)[::-1]

        preporuke_id = []
        vec_dodato = set()

        for idx in top_indeksi:
            s_id = df_pesme.iloc[idx]["song_id"]
            if s_id not in skup_slusanih_id and s_id not in vec_dodato:
                preporuke_id.append(s_id)
                vec_dodato.add(s_id)
            if len(preporuke_id) == top_n:
                break

        # --- RACUNANJE METRIKA PO KORISNIKU ---
        pogoci = [pesma for pesma in preporuke_id if pesma in user_test_pesme]
        broj_pogodaka = len(pogoci)

        if broj_pogodaka > 0:
            ukupno_hitova += 1
        ukupno_preciznost += broj_pogodaka / top_n

        # Računanje Average Precision (za MAP)
        korisnik_ap = 0.0
        trenutni_pogoci = 0
        for i, pesma in enumerate(preporuke_id):
            if pesma in user_test_pesme:
                trenutni_pogoci += 1
                korisnik_ap += trenutni_pogoci / (i + 1)
        
        maksimalno_mogucih_pogodaka = min(top_n, len(user_test_pesme))
        if maksimalno_mogucih_pogodaka > 0:
            ukupno_ap += korisnik_ap / maksimalno_mogucih_pogodaka

        # Računanje NDCG
        korisnik_dcg = 0.0
        for i, pesma in enumerate(preporuke_id):
            if pesma in user_test_pesme:
                korisnik_dcg += 1.0 / np.log2(i + 2)

        korisnik_idcg = 0.0
        for i in range(maksimalno_mogucih_pogodaka):
            korisnik_idcg += 1.0 / np.log2(i + 2)

        if korisnik_idcg > 0:
            ukupno_ndcg += korisnik_dcg / korisnik_idcg

        procenjeni_korisnici += 1

    # (Obrisano duplirano procenjeni_korisnici += 1 koje je stajalo ovde)

    # --- ISPIS REZULTATA ---
    prosecan_hr = ukupno_hitova / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    prosecna_preciznost = ukupno_preciznost / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    prosecan_map = ukupno_ap / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    prosecan_ndcg = ukupno_ndcg / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    
    print(f"\nEvaluirano korisnika (Jaccard Similarity): {procenjeni_korisnici}")
    print(f"Hit Rate@{top_n}:     {prosecan_hr:.4f}")
    print(f"Precision@{top_n}:    {prosecna_preciznost:.4f}")
    print(f"MAP@{top_n}:          {prosecan_map:.4f}")
    print(f"NDCG@{top_n}:         {prosecan_ndcg:.4f}")

    return prosecan_hr, prosecna_preciznost, prosecan_map, prosecan_ndcg


# --- POZIV FUNKCIJE ---
t_matrica = tfidf_matrica if 'tfidf_matrica' in locals() else None
s_maping = song_id_u_indeks if 'song_id_u_indeks' in locals() else None

# Ovde hvatamo sva 4 rezultata koja funkcija vraća da ne dobiješ ValueError
hr_full, prec_full, map_full, ndcg_full = evaluiraj_maksimalni_jaccard(
    df_train=df_train,
    df_test=df_test,  
    tfidf_matrica=t_matrica, 
    song_id_u_indeks=s_maping,
    df_pesme=df_jedinstvene_pesme, 
    top_n=10,
)

Pripremam skupove reči za sve pesme...
Započinjem evaluaciju korisnika...

Evaluirano korisnika (Jaccard Similarity): 14127
Hit Rate@10:     0.0938
Precision@10:    0.0096
MAP@10:          0.0365
NDCG@10:         0.0472


## DICE SIMILARITY

In [66]:
# for idx_p in range(broj_pesama):
#             skup_pesme = pesme_skupovi_reci[idx_p]
#             presek = 2*len(profil_korisnika_skup.intersection(skup_pesme))
#             unija = len(profil_korisnika_skup)+len(skup_pesme)
#             skorovi[idx_p] = presek / unija if unija > 0 else 0


import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
izvor_interakcija=df


def evaluiraj_maksimalni_dice(
    df_train,
    df_test,
    tfidf_matrica,      
    song_id_u_indeks,    
    df_pesme,
    top_n=15,
):
    ukupno_hitova = 0
    ukupno_preciznost = 0.0
    procenjeni_korisnici = 0

    test_users = df_test["user_id"].unique()

    pesme_skupovi_reci = []
    for _, red in df_pesme.iterrows():
        tekst = str(red['artists_clean_text']) + " " + str(red['terms_clean_text'])
        pesme_skupovi_reci.append(set(tekst.split()))
    
    broj_pesama = len(pesme_skupovi_reci)
    brzi_maping = {red['song_id']: i for i, red in df_pesme.iterrows()}

    for target_user_id in test_users:
        user_test_redovi = df_test[df_test["user_id"] == target_user_id]
        if user_test_redovi.empty:
            continue
            
        user_train = df_train[df_train["user_id"] == target_user_id]
        if user_train.empty:
            continue

        user_test_pesme = set(user_test_redovi["song_id"])
        skup_slusanih_id = set(user_train["song_id"])

        # Profil korisnika
        profil_korisnika_skup = set()
        for s_id in user_train["song_id"]:
            if s_id in brzi_maping:
                idx_p = brzi_maping[s_id]
                profil_korisnika_skup.update(pesme_skupovi_reci[idx_p])

        if not profil_korisnika_skup:
            continue

        # Dice
        skorovi = np.zeros(broj_pesama)
        for idx_p in range(broj_pesama):
            skup_pesme = pesme_skupovi_reci[idx_p]
            presek = 2*len(profil_korisnika_skup.intersection(skup_pesme))
            unija = len(profil_korisnika_skup)+len(skup_pesme)
            skorovi[idx_p] = presek / unija if unija > 0 else 0

        top_indeksi = np.argsort(skorovi)[::-1]

        preporuke_id = []
        vec_dodato = set()

        for idx in top_indeksi:
            s_id = df_pesme.iloc[idx]["song_id"]
            if s_id not in skup_slusanih_id and s_id not in vec_dodato:
                preporuke_id.append(s_id)
                vec_dodato.add(s_id)
            if len(preporuke_id) == top_n:
                break

        # Metrike
        pogoci = [pesma for pesma in preporuke_id if pesma in user_test_pesme]
        broj_pogodaka = len(pogoci)

        if broj_pogodaka > 0:
            ukupno_hitova += 1
        ukupno_preciznost += broj_pogodaka / top_n
        procenjeni_korisnici += 1

    prosecan_hr = ukupno_hitova / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    prosecna_preciznost = ukupno_preciznost / procenjeni_korisnici if procenjeni_korisnici > 0 else 0
    
    print(f"\nevaluirano korisnika (dice sim): {procenjeni_korisnici}")
    print(f"Hit Rate@{top_n}:     {prosecan_hr:.4f}")
    print(f"Precision@{top_n}:    {prosecna_preciznost:.4f}")

    return prosecan_hr, prosecna_preciznost


t_matrica = tfidf_matrica if 'tfidf_matrica' in locals() else None
s_maping = song_id_u_indeks if 'song_id_u_indeks' in locals() else None

hr_full, prec_full = evaluiraj_maksimalni_dice(
    df_train=df_train,
    df_test=df_test,  
    tfidf_matrica=t_matrica, 
    song_id_u_indeks=s_maping,
    df_pesme=df_jedinstvene_pesme, 
    top_n=5,
)

Pripremam skupove reči za sve pesme...
Započinjem evaluaciju korisnika...

Evaluirano korisnika Dice Similarity 14127
Hit Rate@5:     0.0691
Precision@5:    0.0141
